In [46]:
print("all ok")

all ok


In [47]:
import os
import sys
from pathlib import Path

print(sys.executable)

if Path.cwd().name == "notebooks":
    os.chdir("..")
print(Path.cwd())

import faiss, langchain_google_genai, pypdf

e:\Development\Projects\pdf-qa-rag\.venv\Scripts\python.exe
e:\Development\Projects\pdf-qa-rag


In [48]:
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY missing. Add it to your .env file.")

PDF_PATH = Path("data/deepseek-r1.pdf")
INDEX_DIR = Path("artifacts/faiss_index")
CHUNK_SIZE, CHUNK_OVERLAP = 1000, 150
TOP_K = 5

print("Key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("PDF exists:", PDF_PATH.exists())

Key loaded: True
PDF exists: True


In [49]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

print(len(embeddings.embed_query("Hello world")), "embedding dimensions")
print(llm.invoke("tell code working in 5 words").text)

3072 embedding dimensions
It runs, so it works.


In [50]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFLoader(str(PDF_PATH)).load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = splitter.split_documents(pages)

print(f"{len(pages)} pages -> {len(chunks)} chunks")
print("Source:", chunks[0].metadata["source"], "| Page:", chunks[0].metadata["page"])
print(chunks[0].page_content[:300])

86 pages -> 297 chunks
Source: data\deepseek-r1.pdf | Page: 0
DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via
Reinforcement Learning
DeepSeek-AI
research@deepseek.com
Abstract
General reasoning represents a long-standing and formidable challenge in artificial intelli-
gence. Recent breakthroughs, exemplified by large language models (LLMs) (Brown e


In [51]:
import time
from langchain_community.vectorstores import FAISS

if INDEX_DIR.exists():
    vector_store = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
else:
    vector_store = FAISS.from_documents(chunks[:80], embeddings)
    for i in range(80, len(chunks), 80):
        time.sleep(65)  # free tier: max 100 embedding requests per minute
        vector_store.add_documents(chunks[i:i + 80])
    vector_store.save_local(str(INDEX_DIR))

retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

In [52]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the context below.
Always answer in the same language as the question.
If the answer is not in the context, say "I don't know, i can answer only based on the document."

Context:
{context}

Question: {question}"""
)


def format_docs(docs):
    return "\n\n".join(
        f"[page {d.metadata.get('page', 0) + 1}] {d.page_content}" for d in docs
    )


chain = prompt | llm | StrOutputParser()


def ask(question: str):
    docs = retriever.invoke(question)
    answer = chain.invoke({"context": format_docs(docs), "question": question})
    pages_used = sorted({d.metadata.get("page", 0) + 1 for d in docs})
    return answer, pages_used


answer, pages_used = ask("What is DeepSeek-R1-Zero and how was it trained?")
print(answer)
print("\nSources: pages", pages_used)

DeepSeek-R1-Zero is a reasoning model that relies exclusively on reinforcement learning without supervised fine-tuning. It was trained using the Group Relative Policy Optimization (GRPO) algorithm. The training process involved a multi-stage approach where the model was trained for a total of 10,400 steps (1.6 training epochs) using a learning rate of 3e-6, a KL coefficient of 0.001, and a sampling temperature of 1. For each question, 16 outputs were sampled, with a maximum length of 32,768 tokens before the 8.2k step and 65,536 tokens afterward. Each training step consisted of 32 unique questions, resulting in a training batch size of 512.

Sources: pages [2, 3, 6]


In [53]:
QUESTION = "What is GRPO?"

answer, pages_used = ask(QUESTION)
print(f"Q: {QUESTION}\n\nAI: {answer}\n\n(pages {pages_used})")

Q: What is GRPO?

AI: GRPO is an optimization method used in the RL stage of LLMs that samples a group of outputs for each question and optimizes the policy model by maximizing a specific objective. Unlike PPO, GRPO foregoes the value model and instead estimates advantages from group scores. Additionally, in GRPO, an unbiased estimator of the KL divergence is directly added to the loss.

(pages [3, 14, 15, 16])
